In [33]:
import copy
import csv
import math
import pandas as pd
import numpy as np
from random import choice
from random import randint
from random import random

In [34]:
class Board:

    def __init__(self):
        self.board = np.full((6, 7), ' ', dtype=str)
        self.empty = 42
        self.record = dict()
        self.record[repr(self)] = 1
        self.state = ' '

    def isLegal(self, isX: bool, move: tuple[int, int]):
        r, c = move[0], move[1] - 1
        if r == 0:
            return ((isX and self.board[5, c] == 'X') or (not isX and self.board[5, c] == 'O'))
        if r == 1:
            return self.board[6 - r, c] == ' '
        return (self.board[7 - r, c] != ' ' and self.board[6 - r, c] == ' ')
    
    def playMove(self, icon: str, move: tuple[int, int]):
        r, c = move[0], move[1] - 1
        if r == 0:
            self.board[1:, c] = self.board[:-1, c]
            self.board[0, c] = ' '
            self.empty += 1
        else:
            self.board[6 - r, c] = icon
            self.empty -= 1
        current_state = repr(self)
        if current_state in self.record:
            if self.record[current_state] == 2:
                self.state = "Game ends on a tie!"
            else:
                self.record[current_state] += 1
        else:
            self.record[current_state] = 1  
        self.checkWin('O' if icon == 'X' else 'X')
        self.checkWin(icon)

    def checkWin(self, icon: str):
        b = (self.board == icon)
        if np.any(b[:, :-3] & b[:, 1:-2] & b[:, 2:-1] & b[:, 3:]):
            self.state = f"Game ends on {icon}'s win!"
            return
        if np.any(b[:-3, :] & b[1:-2, :] & b[2:-1, :] & b[3:, :]):
            self.state = f"Game ends on {icon}'s win!"
            return
        if np.any(b[:-3, :-3] & b[1:-2, 1:-2] & b[2:-1, 2:-1] & b[3:, 3:]):
            self.state = f"Game ends on {icon}'s win!"
            return
        if np.any(b[3:, :-3] & b[2:-1, 1:-2] & b[1:-2, 2:-1] & b[:-3, 3:]):
            self.state = f"Game ends on {icon}'s win!"
            return

    def __str__(self):
        printer = "  -----------------------------\n"
        for i in range(6):
            printer += f"{6 - i} |"
            for j in range(7):
                current = self.board[i, j]
                if current in ('X', 'O'):
                    printer += f" {current} |"
                else:
                    printer += "   |"
            printer += "\n  -----------------------------\n"
        printer += "    1   2   3   4   5   6   7"
        return printer
    
    def __repr__(self):
        return "".join(self.board.ravel())

In [35]:
class Player:

    def __init__(self, isX: bool, type: str):
        self.isX = isX
        self.type = type

    def getPossibleMoves(self, board: Board):
        if (board.empty == 0):
            return ["tie"]
        moves = []
        for i in range(1, 8):
            if (board.board[5][i - 1] == 'X' and self.isX) or (board.board[5][i - 1] == 'O' and not self.isX):
                moves.append((0, i))
            for j in range(1, 7):
                if board.board[6 - j][i - 1] == ' ':
                    moves.append((j, i))
                    break
        return moves

    def turn(self, board: Board, printer=True):
        if printer:
            print(f"{self}'s turn")
            print(f"Possible moves: {self.getPossibleMoves(board)}")
        if self.type == "human":
            while True:
                toParse = input("Enter move coordinates separated by a comma: ").split(",")
                if (board.empty == 0 and len(toParse) == 0 and toParse[0] == "tie"):
                    board.state = "Game ends on a tie!"
                    break
                if len(toParse) != 2:
                    print("Invalid move! Try again.")
                    continue
                move = tuple((int(toParse[0]), int(toParse[1])))
                if move[0] not in range(0, 7) or move[1] not in range(1, 8):
                    print("Invalid move! Try again.")
                    continue
                if board.isLegal(self.isX, move):  # type: ignore
                    board.playMove(str(self), move) # type: ignore
                    return move
                else:
                    print("Invalid move! Try again.")
        elif self.type == "MCTS":
            move = self.mcts_search(board, 10000) 
            board.playMove(str(self), move[0]) # type: ignore
            return move[1] # type: ignore
        elif self.type == "DT":
            print("Decision Tree not yet implemented! Passing turn.")
            moves = self.getPossibleMoves(board)
            if moves != ["tie"]:
                board.playMove(str(self), choice(moves)) # type: ignore

    def __str__(self):
        if self.isX:
             return 'X'
        return 'O'

In [ ]:
print("--- Humano Vs Humano ---")
playerX = Player(True, "human")
playerO = Player(False, "human")
board = Board()

print(board)
while board.state == ' ':
    playerX.turn(board)
    print(board)
    if board.state != ' ':
        break
    playerO.turn(board)
    print(board)
print(board.state)

--- Humano Vs Humano ---
  -----------------------------
6 |   |   |   |   |   |   |   |
  -----------------------------
5 |   |   |   |   |   |   |   |
  -----------------------------
4 |   |   |   |   |   |   |   |
  -----------------------------
3 |   |   |   |   |   |   |   |
  -----------------------------
2 |   |   |   |   |   |   |   |
  -----------------------------
1 |   |   |   |   |   |   |   |
  -----------------------------
    1   2   3   4   5   6   7
X's turn
Possible moves: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7)]
  -----------------------------
6 |   |   |   |   |   |   |   |
  -----------------------------
5 |   |   |   |   |   |   |   |
  -----------------------------
4 |   |   |   |   |   |   |   |
  -----------------------------
3 |   |   |   |   |   |   |   |
  -----------------------------
2 |   |   |   |   |   |   |   |
  -----------------------------
1 | X |   |   |   |   |   |   |
  -----------------------------
    1   2   3   4   5   6   7
O

In [1]:
print("hallo")

hallo


In [ ]:
class MCTSNode:

    def __init__(self, board, move=None, parent=None, constant=1.41, isX=True):
        self.board = copy.deepcopy(board)
        self.move = move
        self.parent = parent
        self.constant = constant
        self.isX = isX
        self.children = []
        self.wins = 0
        self.visits = 0
        self.untriedMoves = Player(isX, "MCTS").getPossibleMoves(self.board)

    def select(self):
        return max(self.children, key=lambda c: (c.wins / c.visits) + self.constant * math.sqrt(math.log(self.visits) / c.visits))
    
    def expand(self):
        move = self.untriedMoves.pop()
        next_board = copy.deepcopy(self.board)
        icon = 'X' if self.isX else 'O'
        next_board.playMove(icon, move)
        child_node = MCTSNode(next_board, move=move, parent=self, constant=self.constant, isX=not self.isX)
        self.children.append(child_node)
        return child_node
    
    def update(self, result):
        self.visits += 1
        if not self.isX: 
            self.wins += result
        else:
            self.wins += (1.0 - result)
        if self.parent:
            self.parent.update(result)

    def is_fully_expanded(self):
        return len(self.untriedMoves) == 0

    def is_terminal(self):
        return self.board.state != ' '

    def rollout(self):
        board_x, board_o = self.to_bitboard(self.board.board)
        current_x_turn = self.isX
        all_cells_mask = 0b111111_111111_111111_111111_111111_111111_111111
        while True:
            moves = self.get_bit_moves(board_x, board_o, current_x_turn)
            if not moves: return 0.5
            move_bit, is_pop = choice(moves)
            if is_pop:
                board_x, board_o = self.apply_bit_pop(board_x, board_o, move_bit)
            else:
                if current_x_turn: board_x |= move_bit
                else: board_o |= move_bit
            x_wins = self.check_bit_win(board_x)
            o_wins = self.check_bit_win(board_o)
            if x_wins and o_wins: return 0.5
            if x_wins: return 1.0
            if o_wins: return 0.0
            if (board_x | board_o) == all_cells_mask: return 0.5
            current_x_turn = not current_x_turn

    def to_bitboard(self, grid):
        board_x = 0
        board_o = 0
        for c in range(7):
            for r in range(6):
                shift = c * 7 + r
                if grid[5-r, c] == 'X':
                    board_x |= (1 << shift)
                elif grid[5-r, c] == 'O':
                    board_o |= (1 << shift)
        return board_x, board_o
    
    def get_bit_moves(self, board_x, board_o, isX):
        moves = []
        occupied = board_x | board_o
        for c in range(7):
            bottom_bit = 1 << (c * 7)
            if isX:
                if board_x & bottom_bit: moves.append((bottom_bit, True))
            else:
                if board_o & bottom_bit: moves.append((bottom_bit, True))
            top_bit = 1 << (c * 7 + 5)
            if not (occupied & top_bit):
                column_mask = 0b111111 << (c * 7)
                empty_in_col = (~occupied) & column_mask
                lowest_empty = empty_in_col & -empty_in_col
                moves.append((lowest_empty, False))
        return moves
    
    def apply_bit_pop(self, board_x, board_o, move_bit):
        col_idx = 0
        temp_bit = move_bit
        while temp_bit > 0b111111:
            temp_bit >>= 7
            col_idx += 1
        col_mask = 0b111111 << (col_idx * 7)
        bits_above_mask = (col_mask ^ move_bit) & (-(move_bit << 1))
        new_x = (board_x & ~col_mask) | ((board_x & bits_above_mask) >> 1)
        new_o = (board_o & ~col_mask) | ((board_o & bits_above_mask) >> 1)
        return new_x, new_o

    def check_bit_win(self, bitboard):
        m = bitboard & (bitboard >> 7)
        if m & (m >> 14): return True
        m = bitboard & (bitboard >> 1)
        if m & (m >> 2): return True
        m = bitboard & (bitboard >> 6)
        if m & (m >> 12): return True
        m = bitboard & (bitboard >> 8)
        if m & (m >> 16): return True
        return False

In [ ]:
def mcts_search(self, rootBoard: Board, iterations=10000):
    if (self.type != "MCTS"):
        return
    deterministic = True
    if randint(0, 1) == 1:
        deterministic = False
    constant = 0.91 + random()
    rootNode = MCTSNode(rootBoard, None, None, constant, self.isX)
    for _ in range(iterations):                   
        node = rootNode
        while node.is_fully_expanded() and not node.is_terminal():
            node = node.select()
        if not node.is_terminal():
            node = node.expand()
        result = node.rollout()
        node.update(result)
    best_move_node = max(rootNode.children, key=lambda c: c.visits)
    if deterministic:
        return best_move_node.move, best_move_node.move
    else:
        visit_counts = [child.visits for child in rootNode.children]
        total_visits = sum(visit_counts)
        draw = random() * total_visits
        id = -1
        while draw > 0:
            id += 1
            draw -= visit_counts[id]
        return rootNode.children[id].move, best_move_node.move